# Entregable 3 — Sistema de 4 agentes sobre Qwen3-4B
**IA Generativa (580694) · UdeC · QA sobre normativa de pregrado (RI-FI, RG, Calendario 2026)**

Este notebook corre el **mismo** sistema de agentes que probamos localmente con un modelo simulado, pero ahora con **Qwen3-4B** de verdad. El código de los agentes vive en archivos del repositorio; los *prompts* de cada agente son archivos versionados aparte (carpeta `prompts/`), como exige el enunciado.

**Qué hace, en orden:** (1) revisa la GPU, (2) instala dependencias, (3) sube el código y los datos (el zip), (4) carga Qwen3-4B en 4 bits, (5) activa el recuperador denso E5, (6) prueba rápida mostrando los mensajes **entre** agentes, (7) corre el benchmark (50 congeladas + 25 difíciles) y guarda respuestas y trazas para **corrección manual**.

> El notebook **no** calcula el accuracy solo: guarda las respuestas con una columna `correcto` vacía para que la corrección sea manual, como hemos hecho en todo el proyecto.

## Paso 0 — Revisar la GPU
Usa un entorno con GPU (Entorno de ejecución → Cambiar tipo → T4).

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memoria total (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).')

## Paso 1 — Instalar dependencias
Qwen3 necesita `transformers` reciente; `bitsandbytes` permite cargar el modelo en 4 bits (cabe en una T4). No se toca `pandas` (Colab depende de su versión).

In [ ]:
# Solo se actualiza lo necesario; pandas y el resto quedan en la versión que trae Colab.
%pip install -q -U "transformers>=4.51" accelerate bitsandbytes
%pip install -q sentence-transformers
print('Listo. Si Colab lo pide, reinicia el entorno (Runtime → Restart) y vuelve a correr desde el Paso 2.')

## Paso 2 — Clonar el repositorio (código, prompts y datos)
El código de los agentes, los prompts versionados y el corpus se descargan directamente del repositorio de GitHub.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/alnicozu/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-2.git'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO], check=True)
print('Commit:', subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
BASE = os.path.join(REPO, 'Deliverable3')
print('Versión del orquestador:', open(os.path.join(BASE, 'agentes', 'orquestador.py'), encoding='utf-8').readline().strip())

PROMPTS = os.path.join(BASE, 'prompts')
AGENTES = os.path.join(BASE, 'agentes')
for ruta in [BASE, AGENTES, PROMPTS]:
    print(('OK  ' if os.path.exists(ruta) else 'FALTA ') + ruta)
assert os.path.exists(AGENTES), 'No encuentro Deliverable3/agentes en el repositorio.'
if AGENTES not in sys.path:
    sys.path.insert(0, AGENTES)
print('\nPrompts versionados:', sorted(f for f in os.listdir(PROMPTS) if f.endswith('.txt')))

## Paso 3 — Cargar Qwen3-4B en 4 bits
Cuantización NF4 (4 bits) con cómputo en bfloat16. La función `generar(agente, prompt)` es la que el orquestador **inyecta** en cada agente: aquí es una llamada real a Qwen3-4B.

Decodificación **igual que en el D2** y como recomienda Qwen para el modo sin razonamiento (`enable_thinking=False`): temperatura 0,7, top-p 0,8, top-k 20, min-p 0. Para que sea **reproducible**, cada llamada fija una semilla derivada de la semilla base del D2 (2026) y del propio prompt: el mismo prompt da siempre la misma salida, y un reintento con la corrección del Verificador (prompt distinto) da una salida nueva.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, hashlib

MODELO = 'Qwen/Qwen3-4B'
# La T4 no tiene soporte nativo de bfloat16: en ese caso se calcula en float16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map='auto')
model.eval()
print('Modelo cargado:', MODELO, '| cómputo en', DTYPE)

# Presupuesto de tokens de salida por agente (el Redactor y el Verificador razonan en <pensar>).
MAX_TOKENS = {'enrutador': 256, 'recuperador': 160, 'redactor': 1200, 'verificador': 700}
SEMILLA = 2026   # misma semilla base que el D2

def generar(agente, prompt, max_new_tokens=None):
    """Llamada real a Qwen3-4B. Firma f(agente, prompt)->str que espera el orquestador."""
    msgs = [{'role':'user','content':prompt}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    max_new_tokens = max_new_tokens or MAX_TOKENS.get(agente, 512)
    semilla = SEMILLA + int(hashlib.sha256((agente + prompt).encode('utf-8')).hexdigest()[:8], 16) % 1_000_000
    torch.manual_seed(semilla)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
                             pad_token_id=tok.eos_token_id)
    gen = out[0][inputs.input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print(generar('prueba', 'Responde solo con la palabra: listo'))

## Paso 4 — Recuperador denso E5
Cargamos `multilingual-e5-small` y precomputamos los vectores de los 201 fragmentos (el campo `texto_embedding` ya viene con el prefijo `passage:`). Reemplazamos el respaldo léxico de la herramienta `buscar` por la búsqueda por coseno, igual que en el Entregable 2.

In [ ]:
from tools import Corpus
from sentence_transformers import SentenceTransformer
import numpy as np

corpus = Corpus(AGENTES)
e5 = SentenceTransformer('intfloat/multilingual-e5-small', device='cuda' if torch.cuda.is_available() else 'cpu')
frag = corpus.frag
P = e5.encode([f['texto_embedding'] for f in frag], normalize_embeddings=True,
              batch_size=64, show_progress_bar=True)

def buscar_e5(consulta, k=5):
    q = e5.encode(['query: ' + str(consulta)], normalize_embeddings=True)[0]
    sims = P @ q
    orden = np.argsort(-sims)
    vistos, salida = set(), []
    for i in orden:
        f = frag[int(i)]
        uid = f.get('unidad_id') or f.get('parent_id')
        if uid in vistos:
            continue
        vistos.add(uid)
        u = corpus.por_id.get(uid, {})
        salida.append({'id': uid, 'texto': u.get('texto', f['texto'])})
        if len(salida) >= k:
            break
    return salida

corpus.buscar = buscar_e5   # inyecta el recuperador denso en la herramienta
print('Recuperador E5 listo. Prueba:')
for h in corpus.buscar('nota minima para aprobar una asignatura', 3):
    print(' ', h['id'], '-', h['texto'][:70])

## Paso 5 — Orquestador + prueba rápida (mensajes entre agentes)
Instanciamos el sistema con el `generar` real y resolvemos **una** pregunta. La traza muestra lo que el video debe enseñar: qué decide el Enrutador, qué acciones y observaciones tiene el Recuperador (ReAct), qué escribe el Redactor y qué decide el Verificador (incluido el chequeo de citas).

In [ ]:
from orquestador import Orquestador
import json

orq = Orquestador(corpus, generar)

def mostrar_traza(r):
    print('TIPO:', r['tipo'], '| ciclos:', r['ciclos'], '| veredicto:', r['veredicto'])
    for p in r['traza']:
        print(f"\n[{p['paso']}] {p['agente'].upper()}" + ('  (fallback)' if p.get('fallback') else ''))
        print('   entrada:', json.dumps(p['entrada'], ensure_ascii=False)[:300])
        sal = p['salida'] if isinstance(p['salida'], str) else json.dumps(p['salida'], ensure_ascii=False)
        print('   salida :', sal[:300])
        if p.get('observacion'):
            print('   obs    :', p['observacion'])
    print('\nRESPUESTA FINAL:\n' + r['respuesta'])

mostrar_traza(orq.responder('¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?'))

Prueba también un caso de **premisa falsa** (debe abstenerse, no inventar). Sirve como caso para el video.

In [ ]:
mostrar_traza(orq.responder('El Artículo 45 del Reglamento Interno de la Facultad de Ingeniería fija la nota mínima de la práctica profesional. ¿Cuál es?'))

## Paso 5d — Casos del video: mensajes entre agentes en vivo

Cada caja aparece **mientras el sistema corre**: la decisión del Enrutador, cada acción ReAct del Recuperador con su
observación, el borrador del Redactor y el veredicto del Verificador (aprobar, devolver al Redactor o re-buscar).
Debajo se muestra la respuesta del RAG del E2 a la misma pregunta (guardada en el repositorio) y la respuesta de referencia.

In [ ]:
import html, json, time
import pandas as pd
from IPython.display import HTML, display
from orquestador import Orquestador

COLOR = {'enrutador': '#22c55e', 'recuperador': '#38bdf8', 'redactor': '#a78bfa', 'verificador': '#f59e0b'}

def _e(t, n=700):
    t = t if isinstance(t, str) else json.dumps(t, ensure_ascii=False)
    return html.escape(t if len(t) <= n else t[:n] + ' …')

def _caja(color, titulo, cuerpo, fondo='#1e293b'):
    display(HTML(f'<div style="background:{fondo};border-left:5px solid {color};padding:8px 12px;margin:5px 0;'
                 f'border-radius:4px;color:#e2e8f0;font-family:ui-monospace,monospace;font-size:13px;line-height:1.45">'
                 f'<b style="color:{color}">{titulo}</b><br>{cuerpo}</div>'))

def mostrar_paso(p):
    a, sal = p['agente'], p['salida']
    titulo = f"[{p['paso']}] {a.upper()}"
    if p.get('decidido_por'):
        titulo += f" (decidido por: {p['decidido_por']})"
    if p.get('fallback'):
        titulo += ' · salida no interpretable → respaldo'
    if a == 'enrutador':
        cuerpo = f"tipo: <b>{_e(sal.get('tipo'))}</b><br>sub-consultas: {_e(sal.get('sub_consultas'))}"
    elif a == 'recuperador':
        cuerpo = f"acción: <b>{_e(sal.get('accion'))}</b> {_e(sal.get('args', ''), 200)}"
        if p.get('corregido'):
            cuerpo += f"<br><span style='color:#fb923c'>orquestador: {_e(p['corregido'])}</span>"
        if p.get('observacion') is not None:
            cuerpo += f"<br>observación: {_e(p['observacion'], 300)}"
    elif a == 'redactor':
        cuerpo = f"modo: {_e(p['entrada'].get('modo'))}"
        if p['entrada'].get('feedback'):
            cuerpo += f"<br>corrige según: <i>{_e(p['entrada']['feedback'], 300)}</i>"
        cuerpo += '<br>borrador:<br>' + _e(sal, 1500).replace('\n', '<br>')
    else:
        destino = sal.get('destino')
        accion = {'fin': 'APRUEBA (fin)', 'redactor': 'DEVUELVE AL REDACTOR', 'recuperador': 'PIDE RE-BUSCAR'}.get(destino, destino)
        if destino == 'fin' and not sal.get('aprobado'):
            accion = 'CIERRA SIN APROBAR'
        cuerpo = (f"chequear_citas: {_e(p['entrada'].get('chequeo_citas'), 300)}<br>"
                  f"decisión: <b>{accion}</b><br>motivo: {_e(sal.get('motivo'), 500)}")
    _caja(COLOR.get(a, '#94a3b8'), titulo, cuerpo)

class OrquestadorEnVivo(Orquestador):
    """Mismo orquestador; además dibuja cada paso apenas se registra en la traza."""
    def _log(self, agente, entrada, salida, extra=None):
        super()._log(agente, entrada, salida, extra)
        mostrar_paso(self.traza[-1])

BASE50 = pd.read_csv(os.path.join(AGENTES, 'test_set_50.csv'))
DIF25 = pd.read_csv(os.path.join(AGENTES, 'test_set_dificil_25.csv'))
D2_DIF = pd.read_csv(os.path.join(BASE, 'resultados_d2_dificiles', 'rag_estructurado.csv'))
TRAZAS_VIDEO = {}

def caso(set_, qid):
    if set_ == 'dificil_25':
        fila = DIF25[DIF25['id'].astype(str) == str(qid)].iloc[0]
        d2 = D2_DIF[D2_DIF['pregunta'] == fila['pregunta']]['respuesta_modelo']
    else:
        fila = BASE50[BASE50['id'] == int(qid)].iloc[0]
        d2 = pd.Series(dtype=str)
    _caja('#e2e8f0', f'PREGUNTA {qid} · {fila["categoria"]}', '<i>' + _e(fila['pregunta']) + '</i>', '#0f172a')
    t0 = time.time()
    r = OrquestadorEnVivo(corpus, generar).responder(str(fila['pregunta']))
    ok = r['veredicto'].get('aprobado')
    _caja('#22c55e' if ok else '#ef4444',
          f"RESPUESTA FINAL · {'aprobada por el Verificador' if ok else 'NO aprobada'} · {r['ciclos']} ciclo(s) · "
          f"{len(r['traza'])} llamadas · {time.time() - t0:.0f} s",
          _e(r['respuesta'], 2000).replace('\n', '<br>'), '#052e1a' if ok else '#3b0a0a')
    if len(d2):
        _caja('#94a3b8', 'RAG DEL E2 (misma pregunta, respuesta guardada en el repositorio)', _e(d2.iloc[0], 1200))
    _caja('#94a3b8', 'REFERENCIA (pauta; no se entrega al sistema)', _e(f"{fila['gold_dato']} ({fila['gold_fuente']})"))
    TRAZAS_VIDEO[f'{set_}:{qid}'] = r['traza']
    return r

print('Visor en vivo listo.')

**H06 (cálculo).** 38 créditos al cuarto semestre, ¿cae en baja? Hay que dividir 38/4 = 9,5 < 10. El RAG del E2 falló;
en la corrida evaluada el Verificador devolvió dos veces el borrador al Redactor antes de aprobarlo.

In [ ]:
_ = caso('dificil_25', 'H06')

**P25 (el caso de falla del Entregable 2).** ¿Qué asignaturas no se pueden eliminar al modificar la inscripción?
El E2 respondió solo «prioridades a) y b)». Aquí el Verificador puede pedir al Recuperador que re-busque.

In [ ]:
_ = caso('congelado_50', 25)

**H20 (premisa falsa).** El Artículo 45 del RI-FI no existe: el Enrutador lo clasifica como premisa falsa y el sistema
verifica la existencia del artículo con la herramienta, sin buscar ni inventar.

In [ ]:
_ = caso('dificil_25', 'H20')

**H03 (caso de falla).** Días entre el inicio de clases y la fecha límite de modificación en cada semestre. En la
corrida evaluada las fechas de inicio no llegaron a la evidencia; el Verificador rechazó el cálculo inventado y mandó
a re-buscar, pero el Recuperador repitió búsquedas parecidas y la respuesta salió **no aprobada**.

In [ ]:
_ = caso('dificil_25', 'H03')

In [ ]:
# Guarda las trazas de los casos del video (subirlas a Deliverable3/trazas_video/ en el repositorio).
with open('trazas_video_D3.json', 'w', encoding='utf-8') as f:
    json.dump(TRAZAS_VIDEO, f, ensure_ascii=False, indent=1)
print('Guardado trazas_video_D3.json con', list(TRAZAS_VIDEO))
try:
    from google.colab import files
    files.download('trazas_video_D3.json')
except Exception as e:
    print('Descárgalo desde el panel de archivos:', e)

## Paso 5c — Elegir la variante (ablaciones)
`ABLACION = None` corre el sistema completo (v6). Para medir cuánto aporta cada técnica, se quita **una** a la vez:

| `ABLACION` | Agente | Qué se quita |
|---|---|---|
| `'sin_fewshot'` | Enrutador | los ejemplos few-shot |
| `'sin_react'` | Recuperador | el ciclo ReAct (queda solo la búsqueda base con la pregunta) |
| `'sin_cot'` | Redactor | la instrucción de razonar paso a paso antes de responder |
| `'sin_verificador'` | Verificador | el bucle de revisión (se entrega el primer borrador) |

`SOLO_DIFICILES = True` corre solo las 25 preguntas difíciles (el set de las ablaciones). Cada variante guarda en su propia carpeta de Drive (`D3_ablacion_<nombre>`). **Ejecuta esta celda antes del 5b.**

In [ ]:
ABLACION = 'sin_fewshot'     # None, 'sin_fewshot', 'sin_react', 'sin_cot' o 'sin_verificador'
SOLO_DIFICILES = True          # True: solo las 25 difíciles (set de ablaciones)

from orquestador import ABLACIONES
orq = Orquestador(corpus, generar, ablacion=ABLACION)
print('Variante:', ABLACION or 'sistema completo (v6)', '|', ABLACIONES.get(ABLACION, 'todas las técnicas activas'))

## Paso 5b — Guardar en Google Drive
Los resultados se guardan en tu Drive (carpeta `D3_resultados_v6`; las corridas anteriores quedaron en `D3_resultados` y `D3_resultados_v5`), no en el disco temporal de Colab: si Colab se desconecta, el avance no se pierde. Al correr la celda, Google pide autorizar el acceso.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
CARPETA_RESULTADOS = ('/content/drive/MyDrive/D3_ablacion_' + ABLACION) if ABLACION else '/content/drive/MyDrive/D3_resultados_v6'
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
os.chdir(CARPETA_RESULTADOS)   # el Paso 6 guarda aquí
print('Los resultados se guardarán en:', os.getcwd())
print('Ya hay:', os.listdir('.'))

## Paso 6 — Corrida (set según `SOLO_DIFICILES`; variante según `ABLACION`)
Corre el sistema sobre los dos sets. **Guarda después de cada pregunta**: si Colab se desconecta, vuelve a correr los Pasos 0, 2, 3, 4, la primera celda del 5, el 5c, el 5b y esta celda: continúa donde quedó (no repite las ya hechas).

**No se calcula el accuracy aquí**: la columna `correcto` queda vacía para corrección manual (0/1), como en D1 y D2. Gracias a la semilla por llamada, volver a correr da las mismas respuestas.

In [ ]:
import pandas as pd, json, time, os

SALIDA_CSV, SALIDA_TRAZAS = 'resultados_agentes_D3.csv', 'trazas_D3.json'
sets = [('congelado_50', os.path.join(AGENTES, 'test_set_50.csv')),
        ('dificil_25',  os.path.join(AGENTES, 'test_set_dificil_25.csv'))]
if SOLO_DIFICILES:
    sets = [s for s in sets if s[0] == 'dificil_25']
print('Variante:', ABLACION or 'sistema completo (v6)', '| sets:', [s[0] for s in sets])

# Retomar si ya hay avance guardado
filas = pd.read_csv(SALIDA_CSV).to_dict('records') if os.path.exists(SALIDA_CSV) else []
trazas = json.load(open(SALIDA_TRAZAS, encoding='utf-8')) if os.path.exists(SALIDA_TRAZAS) else {}
hechas = {f"{f['set']}:{f['id']}" for f in filas}
print('Ya respondidas:', len(hechas))

def fuentes_de(traza):
    out = set()
    for p in traza:
        obs = p.get('observacion') or {}
        if p['agente'] == 'recuperador':
            if obs.get('traido'): out.add(obs['traido'])
            out.update(obs.get('hits', []))
    return '; '.join(sorted(out))

t_inicio = time.time()
for nombre_set, ruta in sets:
    df = pd.read_csv(ruta)
    print(f'\n=== {nombre_set}: {len(df)} preguntas ===')
    for _, row in df.iterrows():
        qid = f"{nombre_set}:{row['id']}"
        if qid in hechas:
            continue
        t0 = time.time()
        try:
            res = orq.responder(str(row['pregunta']))
            error = ''
        except Exception as e:
            res = {'respuesta': '', 'veredicto': {}, 'ciclos': 0, 'tipo': '', 'traza': list(orq.traza)}
            error = repr(e)
        traza = res['traza']
        filas.append({
            'set': nombre_set, 'ablacion': ABLACION or 'completo', 'id': row['id'], 'categoria': row.get('categoria', ''),
            'pregunta': row['pregunta'], 'gold_dato': row.get('gold_dato', ''), 'gold_fuente': row.get('gold_fuente', ''),
            'respuesta_sistema': res['respuesta'], 'tipo_enrutador': res['tipo'],
            'aprobado_verificador': res['veredicto'].get('aprobado', ''), 'motivo_verificador': res['veredicto'].get('motivo', ''),
            'ciclos': res['ciclos'], 'llamadas_modelo': len(traza),
            'fallbacks': sum(1 for p in traza if p.get('fallback')),
            'fuentes_traidas': fuentes_de(traza), 'segundos': round(time.time() - t0, 1),
            'error': error, 'correcto': ''})   # <-- 'correcto' se llena a mano (0/1)
        trazas[qid] = traza
        pd.DataFrame(filas).to_csv(SALIDA_CSV, index=False)
        json.dump(trazas, open(SALIDA_TRAZAS, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        print(f"  {qid:18s} {res['tipo']:13s} ciclos={res['ciclos']}  {round(time.time()-t0,1)}s {('ERROR '+error) if error else ''}")

res_df = pd.DataFrame(filas)
print(f'\nListo: {len(res_df)} filas en {SALIDA_CSV} ({round((time.time()-t_inicio)/60,1)} min en esta sesión).')

### Resumen del comportamiento (no es el accuracy)
Estas cifras describen **cómo trabajó** el sistema: cuántas veces el Verificador mandó a corregir, cuántas salidas del modelo no se pudieron interpretar (fallbacks) y cuánto tardó. La exactitud real sale de la corrección manual de la columna `correcto`.

In [ ]:
res_df = pd.read_csv(SALIDA_CSV)
print(res_df.groupby('set').agg(preguntas=('id','count'), ciclos_prom=('ciclos','mean'),
      con_correccion=('ciclos', lambda c: int((c > 1).sum())), fallbacks=('fallbacks','sum'),
      seg_prom=('segundos','mean'), errores=('error', lambda e: int(e.fillna('').astype(bool).sum()))).round(2))
print('\nTipos asignados por el Enrutador:')
print(res_df.groupby(['set','tipo_enrutador']).size())

## Paso 7 — Descargar resultados y trazas
`resultados_agentes_D3.csv` (para corregir a mano) y `trazas_D3.json` (mensajes entre agentes, para el repositorio).

In [ ]:
try:
    from google.colab import files
    files.download(SALIDA_CSV)
    files.download(SALIDA_TRAZAS)
except Exception as e:
    print('Descarga manual desde el panel de archivos. Detalle:', e)

---
### Qué sigue (Paso 3 del plan)
Con el sistema completo funcionando, el siguiente paso es la **ablación**: correr variantes quitando una técnica de prompting a la vez (p. ej. sin few-shot en el Enrutador, sin CoT en el Redactor, sin la herramienta `existe_articulo` en el Verificador) y comparar el accuracy **manual** de cada variante contra el sistema completo. Eso mide cuánto aporta cada técnica, como pide la rúbrica.